# Pipeline Recipe Requirements Check / Pipeline Recipe 要件確認

This notebook is a lightweight acceptance check for the current Pipeline Recipe requirements. It uses small synthetic frames, no downloads, and assert-driven cells so a developer can see which requirement each behavior supports.
このNotebookは、現在の Pipeline Recipe 要件を確認するための軽量な受け入れチェックです。小さな合成Frameだけを使い、外部データのダウンロードは行わず、各セルのassertからどの要件を確認しているかを追えるようにしています。

| Requirement / 要件 | Checked here / 確認内容 |
| --- | --- |
| R1 | `OperationSpec` stores a replayable operation key and stable params. / `OperationSpec` がreplay可能なoperation keyと安定したparamsを保持する。 |
| R2 | `RecipeSpec.apply(frame)` applies multiple operations in order. / `RecipeSpec.apply(frame)` が複数operationを順番通りに適用する。 |
| R3 | Applying a recipe does not mutate the input frame. / Recipe適用で入力Frameを破壊しない。 |
| R4 | Dask-backed frames remain lazy until explicit compute. / Dask-backed frameが明示的なcomputeまでlazyのまま残る。 |
| R5 | sklearn-compatible transformers support `fit`, `transform`, `get_params`, `set_params`, and `to_spec` when sklearn is installed. / sklearn導入時、互換transformerが `fit`, `transform`, `get_params`, `set_params`, `to_spec` を提供する。 |
| R6 | Representative signal-operation transformer classes are available when sklearn is installed. / sklearn導入時、代表的な信号処理transformer classを利用できる。 |
| R7 | Normal `import wandas` does not import sklearn. / 通常の `import wandas` ではsklearnをimportしない。 |
| R8 | Missing sklearn produces an ImportError that points to `pip install "wandas[sklearn]"`. / sklearn未導入時は `pip install "wandas[sklearn]"` を案内するImportErrorになる。 |
| R9 | `RecipeSpec.from_frame(processed)` extracts a replayable linear recipe from exploratory frame work. / 探索的に処理したframeから `RecipeSpec.from_frame(processed)` で直列Recipeを抽出できる。 |
| R10 | `GraphRecipeSpec.from_frame(...)` extracts a replayable named-input graph recipe. / `GraphRecipeSpec.from_frame(...)` で名前付き入力のgraph recipeを抽出できる。 |
| R11 | Explicit method, indexing, scalar, custom, and terminal step types replay through existing frame APIs. / 明示step型が既存frame APIへ委譲してreplayできる。 |
| R12 | Unsupported extraction boundaries raise `RecipeExtractionError`. / 対応外の抽出境界は `RecipeExtractionError` で明示される。 |

## Shared Test Fixtures / 共通テストフィクスチャ

In [ ]:
import json
import importlib
import subprocess
import sys
import tempfile
from pathlib import Path

import dask.array as da
import numpy as np
from dask.array.core import Array as DaArray
from dask.delayed import delayed

from wandas.frames.channel import ChannelFrame
from wandas.pipeline import (
    CustomFunctionStep,
    GraphRecipeSpec,
    IndexingStep,
    MethodStep,
    NodeGraphRecipeSpec,
    OperationSpec,
    RecipeExtractionError,
    RecipeSpec,
    ScalarOperationStep,
    TerminalStep,
    TypedMethodStep,
)


def make_frame(label="source"):
    sampling_rate = 8000
    time = np.linspace(0.0, 0.1, int(sampling_rate * 0.1), endpoint=False)
    data = (0.25 + np.sin(2 * np.pi * 440 * time)).reshape(1, -1)
    return ChannelFrame.from_numpy(data, sampling_rate=sampling_rate, label=label)


frame = make_frame()
other_frame = make_frame("other")

assert frame.shape == other_frame.shape
assert frame.operation_history == []
print("fixtures ready")

## R1: OperationSpec Represents One Replayable Operation / replay可能な1つのoperationを表す

`OperationSpec` stores the operation registry key and an immutable parameter snapshot. The operation key is the processing registry name, such as `normalize`, not necessarily the user-facing method name.
`OperationSpec` はoperation registry keyと変更されないparameter snapshotを保持します。operation keyは `normalize` のようなprocessing registry名であり、必ずしもユーザー向けmethod名とは一致しません。

In [ ]:
step = OperationSpec("normalize", {"norm": 2.0, "axis": -1, "threshold": None, "fill": None})

assert step.operation == "normalize"
assert step.params == {"norm": 2.0, "axis": -1, "threshold": None, "fill": None}
assert step.to_dict() == {
    "operation": "normalize",
    "params": {"norm": 2.0, "axis": -1, "threshold": None, "fill": None},
}

mutable_copy = step.params
mutable_copy["norm"] = 999.0
assert step.params["norm"] == 2.0
print("R1 passed")

## R2 and R3: Serial Replay and Input Immutability / 直列replayと入力不変性

In [ ]:
recipe = RecipeSpec(
    [
        OperationSpec("remove_dc"),
        OperationSpec("normalize", {"norm": 2.0, "axis": -1, "threshold": None, "fill": None}),
    ]
)

before_data = frame.data.copy()
before_metadata = dict(frame.metadata)
before_history = list(frame.operation_history)

replayed = recipe.apply(frame)

assert [record["operation"] for record in replayed.operation_history] == ["remove_dc", "normalize"]
np.testing.assert_allclose(frame.data, before_data)
assert frame.metadata == before_metadata
assert frame.operation_history == before_history
assert replayed is not frame
print("R2/R3 passed")

## R4: Dask Laziness Is Preserved / Dask lazinessを維持する

Applying the recipe to a delayed Dask-backed frame must not call `.compute()` or evaluate delayed data.
delayedなDask-backed frameへRecipeを適用しても、`.compute()` の呼び出しやdelayed dataの評価は発生しない必要があります。

In [ ]:
calls = []


def build_lazy_data():
    calls.append("computed")
    return np.arange(32, dtype=float).reshape(1, -1)


lazy_data = da.from_delayed(delayed(build_lazy_data)(), shape=(1, 32), dtype=float)
lazy_frame = ChannelFrame(data=lazy_data, sampling_rate=8000, label="lazy")

lazy_replayed = recipe.apply(lazy_frame)

assert calls == []
assert isinstance(lazy_replayed._data, DaArray)
assert lazy_replayed.shape == lazy_frame.shape
print("R4 passed")

## R7: Normal Wandas Import Does Not Import sklearn / 通常importではsklearnを読まない

This check runs in a fresh Python process so it is independent of this notebook kernel state.
この確認は新しいPython processで実行するため、Notebook kernelの現在のimport状態に依存しません。

In [ ]:
code = """
import json
import sys
import wandas
print(json.dumps(any(name == 'sklearn' or name.startswith('sklearn.') for name in sys.modules)))
"""
result = subprocess.run([sys.executable, "-c", code], check=True, text=True, capture_output=True)
assert json.loads(result.stdout) is False
print("R7 passed")

## R5 and R6: sklearn Adapter When sklearn Is Installed / sklearn導入時のadapter

The adapter is optional. If scikit-learn is not installed, this cell reports a skip instead of failing the notebook. R8 below checks the missing-dependency error path directly.
adapterはoptionalです。scikit-learnが未導入の場合、このセルはNotebook全体を失敗させずskipを表示します。未導入時のerror pathは下のR8で直接確認します。

In [ ]:
try:
    from sklearn.pipeline import Pipeline
except ImportError as exc:
    print(f"R5/R6 skipped: {exc}")
else:
    from wandas.pipeline.sklearn import (
        BandPassFilter,
        HighPassFilter,
        LowPassFilter,
        Normalize,
        RemoveDC,
        WandasOperationTransformer,
    )

    hp = HighPassFilter(cutoff=100.0, order=2)
    assert hp.get_params() == {"cutoff": 100.0, "order": 2}
    assert hp.set_params(order=4) is hp
    assert hp.get_params()["order"] == 4
    assert hp.fit(frame) is hp
    assert hp.to_spec() == OperationSpec("highpass_filter", {"cutoff": 100.0, "order": 4})

    assert LowPassFilter(1000.0).to_spec().operation == "lowpass_filter"
    assert BandPassFilter(100.0, 1000.0).to_spec().operation == "bandpass_filter"

    sklearn_pipeline = Pipeline([("dc", RemoveDC()), ("norm", Normalize(norm=2.0))])
    sklearn_replayed = sklearn_pipeline.transform(frame)
    native_replayed = recipe.apply(frame)

    assert sklearn_replayed.operation_history == native_replayed.operation_history
    np.testing.assert_allclose(sklearn_replayed.data, native_replayed.data)

    generic = WandasOperationTransformer("remove_dc")
    assert generic.get_params() == {"operation": "remove_dc"}
    assert generic.to_spec() == OperationSpec("remove_dc")
    print("R5/R6 passed")

## R8: Missing sklearn Error Mentions the Extra / sklearn未導入時にextraを案内する

This check simulates a missing `sklearn.base` import in a fresh process and confirms the error points users to `wandas[sklearn]`.
この確認では、新しいprocess内で `sklearn.base` import失敗をsimulateし、error messageが `wandas[sklearn]` を案内することを確認します。

In [ ]:
code = r'''
import importlib
import sys

real_import_module = importlib.import_module

def blocked_import_module(module_name, package=None):
    if module_name == "sklearn.base":
        raise ModuleNotFoundError("No module named 'sklearn'", name="sklearn")
    return real_import_module(module_name, package)

importlib.import_module = blocked_import_module
sys.modules.pop("wandas.pipeline.sklearn", None)

try:
    importlib.import_module("wandas.pipeline.sklearn")
except ImportError as exc:
    message = str(exc)
    assert "Wandas sklearn transformers requires optional dependency" in message
    assert "pip install \"wandas[sklearn]\"" in message
    print("R8 passed")
else:
    raise AssertionError("expected ImportError for missing sklearn.base")
'''

result = subprocess.run([sys.executable, "-c", code], check=True, text=True, capture_output=True)
print(result.stdout.strip())

## R9: Extract a Linear Recipe from a Processed Frame / 処理済みframeから直列Recipeを抽出する

`RecipeSpec.from_frame(processed)` reads `operation_graph`, not Python source code. It should extract supported exploratory frame operations and replay them on another frame.
`RecipeSpec.from_frame(processed)` はPythonソースコードではなく `operation_graph` を読みます。対応済みの探索的frame操作を抽出し、別のframeへreplayできる必要があります。

In [ ]:
exploratory = frame.remove_dc().normalize(norm=2.0)
extracted = RecipeSpec.from_frame(exploratory)
expected_other = other_frame.remove_dc().normalize(norm=2.0)
replayed_other = extracted.apply(other_frame)

assert extracted.steps == recipe.steps
assert replayed_other.operation_history == expected_other.operation_history
np.testing.assert_allclose(replayed_other.data, expected_other.data)
print("R9 passed")

## R10: Extract a Named-Input Graph Recipe / 名前付き入力のgraph recipeを抽出する

Graph extraction keeps multi-input replay separate from single-input `RecipeSpec`. The caller supplies stable input names instead of relying on Python variable names. This cell checks the readable two-input `GraphRecipeSpec` path plus `NodeGraphRecipeSpec` for an external array operand and an `add_channel` graph.
Graph抽出は、複数入力replayを単一入力の `RecipeSpec` から分離します。Python変数名に依存せず、呼び出し側が安定した入力名を渡します。このセルでは、読みやすい2入力の `GraphRecipeSpec` 経路に加え、外部array operandと `add_channel` graphを `NodeGraphRecipeSpec` で確認します。

In [ ]:
left_processed = frame.remove_dc()
right_processed = other_frame.normalize(norm=2.0)
mixed = left_processed + right_processed

graph_recipe = GraphRecipeSpec.from_frame(mixed, input_names=("signal", "noise"))
graph_replayed = graph_recipe.apply({"signal": frame, "noise": other_frame})

np.testing.assert_allclose(graph_replayed.data, mixed.data)
assert graph_replayed.operation_history == mixed.operation_history

operand = np.ones(frame.shape) * 0.1
operand_processed = frame + operand
operand_recipe = NodeGraphRecipeSpec.from_frame(operand_processed, input_names=("signal", "offset"))
operand_replayed = operand_recipe.apply({"signal": frame, "offset": operand})
np.testing.assert_allclose(operand_replayed.data, operand_processed.data)
assert operand_replayed.operation_history == operand_processed.operation_history

base = ChannelFrame.from_numpy(frame.data, sampling_rate=frame.sampling_rate, ch_labels=["left"])
added = ChannelFrame.from_numpy(other_frame.data * 0.5, sampling_rate=other_frame.sampling_rate, ch_labels=["right"])
added_processed = base.add_channel(added, label="ref")
add_channel_recipe = NodeGraphRecipeSpec.from_frame(added_processed, input_names=("base", "added"))
add_channel_replayed = add_channel_recipe.apply({"base": base, "added": added})
np.testing.assert_allclose(add_channel_replayed.data, added_processed.data)
assert add_channel_replayed.operation_history == added_processed.operation_history
print("R10 passed")

## R11: Explicit Step Types Delegate to Frame APIs / 明示step型はframe APIへ委譲する

Some replayable behavior is represented by dedicated step classes instead of generic operation keys. This cell uses method, typed transition, indexing, scalar arithmetic, importable custom function, and terminal metric steps without duplicating the frame calculation logic in the recipe.
一部のreplay可能な挙動は、generic operation keyではなく専用step classで表現します。このセルでは、Recipe側で計算ロジックを複製せず、method step、typed transition、indexing、scalar演算、importable custom function、terminal metricを既存frame APIへ委譲します。

In [ ]:
module_dir = Path(tempfile.mkdtemp(prefix="wandas_recipe_requirements_"))
sys.path.insert(0, str(module_dir))
(module_dir / "recipe_requirement_ops.py").write_text(
    "def scale(data, gain):\n"
    "    return data * gain\n\n"
    "def same_shape(shape):\n"
    "    return shape\n",
    encoding="utf-8",
)
recipe_requirement_ops = importlib.import_module("recipe_requirement_ops")

explicit_steps = RecipeSpec([
    MethodStep("fix_length", {"length": frame.shape[-1]}),
    IndexingStep(slice(0, 1)),
    ScalarOperationStep("*", 2.0),
    CustomFunctionStep(
        "recipe_requirement_ops.scale",
        {"gain": 0.5},
        output_shape_function="recipe_requirement_ops.same_shape",
    ),
    TerminalStep("rms"),
])
explicit_result = explicit_steps.apply(frame)
expected_result = (frame.fix_length(length=frame.shape[-1])[0:1] * 2.0).apply(
    recipe_requirement_ops.scale,
    output_shape_func=recipe_requirement_ops.same_shape,
    gain=0.5,
).rms

np.testing.assert_allclose(explicit_result, expected_result)

typed_terminal_steps = RecipeSpec([
    TypedMethodStep("fft", {"n_fft": 128, "window": "hann"}),
    TerminalStep("magnitude"),
])
np.testing.assert_allclose(typed_terminal_steps.apply(frame), frame.fft(n_fft=128, window="hann").magnitude)
print("R11 passed")

## R12: Unsupported Callable Extraction Fails Loudly / 未対応callable抽出は明示的に失敗する

Unsupported extraction should fail loudly instead of returning a partial recipe. This cell checks callable, channel-query, and graph boundaries; the full boundary matrix lives in the extraction-boundaries document and unit tests.
未対応の抽出は、部分的なRecipeを返さず明示的に失敗する必要があります。このセルではcallable、channel query、graph境界を確認します。完全な境界matrixは抽出境界資料と単体テストにあります。

In [ ]:
bad = frame.apply(
    lambda data, gain: data * gain,
    output_shape_func=lambda shape: shape,
    gain=2.0,
)

try:
    RecipeSpec.from_frame(bad)
except RecipeExtractionError as exc:
    assert "Custom operation recipe extraction requires importable" in str(exc)
else:
    raise AssertionError("lambda custom operation should not be recipe-extractable")

two_channel = ChannelFrame.from_numpy(
    np.vstack([frame.data, frame.data * 0.5]),
    sampling_rate=frame.sampling_rate,
    ch_labels=["left", "right"],
)
bad_query = two_channel.get_channel(query=lambda channel: channel.label == "right")
try:
    RecipeSpec.from_frame(bad_query)
except RecipeExtractionError as exc:
    assert "Channel selection recipe extraction only supports" in str(exc)
else:
    raise AssertionError("callable channel query should not be recipe-extractable")

bad_graph = frame + other_frame
try:
    RecipeSpec.from_frame(bad_graph)
except RecipeExtractionError as exc:
    assert "RecipeSpec.from_frame(...) cannot extract graph lineage as a linear recipe" in str(exc)
else:
    raise AssertionError("multi-input graph should use a graph recipe class")

print("R12 passed")